# TikTok: predecir si una cuenta está verificada

**Curso 4 del certificado, proyecto de regresión logística binomial.**

TikTok quiere descongestionar la cola de reclamaciones pendientes de revisión. El objetivo
final, que llega en el Curso 5, es clasificar automáticamente si un vídeo es una
reclamación o una opinión. Este proyecto es el paso previo: montar la logística completa
sobre una variable binaria conocida, `verified_status`.

El aviso va por delante porque decide cómo se mide todo: **el 93,7 % de las cuentas no
están verificadas**.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "projects" / "curso4").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "projects"))
sys.path.insert(0, str(ROOT / "projects" / "curso4" / "tiktok" / "02_scripts"))

import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

from tiktok_logistic import balance, encode, load_and_clean, vif_table
from common import Results

pd.set_option("display.width", 120)
print("listo")

listo


## 1. Los datos, y el desbalance que lo gobierna todo

298 filas tienen datos ausentes y se eliminan. Lo que queda es el problema de siempre en
clasificación: una clase aplasta a la otra.

In [2]:
results = Results("tiktok", "TikTok, cuaderno")
df = encode(load_and_clean(results))
df[["verified_status", "claim_status", "author_ban_status", "video_duration_sec"]].head()


1. Los datos, y qué hubo que arreglar
  filas en el CSV: 19382
  filas con algún dato ausente: 298
  filas duplicadas: 0
  filas completas: 19084
  cuentas no verificadas: 17884
  cuentas verificadas: 1200
  porcentaje de verificadas: 6.3
  acierto de responder siempre «no verificada»: 93.7
  % de opiniones entre las verificadas: 82.6
  % de opiniones entre las no verificadas: 47.4
  duración media, verificadas (s): 31.77
  duración media, no verificadas (s): 32.47


  verified_status claim_status author_ban_status  video_duration_sec
0    not verified        claim      under review                  59
1    not verified        claim            active                  32
2    not verified        claim            active                  31
3    not verified        claim            active                  25
4    not verified        claim            active                  19


![Desbalance de clases](03_figures/01_desbalance.png)

## 2. Dónde está la señal

El material del curso apunta a la duración del vídeo. En estos datos la duración apenas
distingue a los dos grupos, y lo que sí los distingue es qué tipo de vídeo publican.

In [3]:
print("duracion media en segundos")
print(df.groupby("verified_status").video_duration_sec.mean().round(2).to_string())

print("\nreparto de reclamaciones y opiniones, por columna")
print((pd.crosstab(df.claim_status, df.verified_status, normalize="columns") * 100)
      .round(1).to_string())

duracion media en segundos
verified_status
not verified    32.47
verified        31.78

reparto de reclamaciones y opiniones, por columna
verified_status  not verified  verified
claim_status                           
claim                    52.6      17.4
opinion                  47.4      82.6


![La señal real](03_figures/02_senal.png)

## 3. Equilibrar las clases

La clase mayoritaria se submuestrea hasta igualar a la minoritaria. Sin esto, el camino más
corto hacia una pérdida baja es no predecir nunca «verificada».

**Consecuencia que hay que tener presente al leer los resultados:** a partir de aquí la
exactitud se compara contra el 50 % de una moneda, no contra el 93,7 % del dataset.

In [4]:
balanced = balance(df, results)
print(balanced.verified.value_counts().to_string())

  filas tras equilibrar: 2400
  filas de la clase mayoritaria descartadas: 16684
verified
0    1200
1    1200


## 4. Multicolinealidad entre los cinco contadores

Visualizaciones, me gusta, compartidos, descargas y comentarios miden en buena medida lo
mismo. Ninguno llega al umbral de 10, así que quedarse con uno fue un criterio propio.

In [5]:
everything = ["video_duration_sec", "is_claim", "banned", "under_review",
              "video_view_count", "video_like_count", "video_share_count",
              "video_download_count", "video_comment_count"]
print(vif_table(balanced[everything]).to_string(index=False,
                                                float_format=lambda v: f"{v:.2f}"))

            variable  vif
    video_like_count 7.76
video_download_count 6.30
    video_view_count 5.19
 video_comment_count 4.05
   video_share_count 3.77
            is_claim 3.04
              banned 1.08
        under_review 1.07
  video_duration_sec 1.00


![Multicolinealidad](03_figures/03_multicolinealidad.png)

## 5. Ajustar la logística y leer los coeficientes en momios

In [6]:
features = ["video_duration_sec", "is_claim", "banned", "under_review", "video_view_count"]
train, test = train_test_split(balanced, test_size=0.25, random_state=42,
                               stratify=balanced.verified)
model = sm.Logit(train.verified, sm.add_constant(train[features])).fit(disp=False)

table = pd.DataFrame({
    "coeficiente": model.params.round(4),
    "razon de momios": np.exp(model.params).round(3),
    "p": model.pvalues.map(lambda v: f"{v:.3g}"),
})
print(table.to_string())
print(f"\npseudo R2 = {model.prsquared:.4f}")

                    coeficiente  razon de momios         p
const                    0.6294            1.876  1.35e-07
video_duration_sec      -0.0015            0.998      0.63
is_claim                -1.6331            0.195  1.92e-16
banned                  -0.2759            0.759     0.234
under_review            -0.2421            0.785     0.196
video_view_count         0.0000            1.000     0.886

pseudo R2 = 0.1046


Solo `is_claim` tiene efecto demostrado. Los demás intervalos cruzan el 1, que es la línea
de no efecto cuando se habla de momios.

![Razones de momios](03_figures/04_momios.png)

## 6. Qué tal clasifica

In [7]:
probability = model.predict(sm.add_constant(test[features]))
predicted = (probability >= 0.5).astype(int)

print(confusion_matrix(test.verified, predicted))
print()
print(classification_report(test.verified, predicted,
                            target_names=["no verificada", "verificada"], digits=3))

[[161 139]
 [ 52 248]]

               precision    recall  f1-score   support

no verificada      0.756     0.537     0.628       300
   verificada      0.641     0.827     0.722       300

     accuracy                          0.682       600
    macro avg      0.698     0.682     0.675       600
 weighted avg      0.698     0.682     0.675       600


![Matriz de confusión](03_figures/05_matriz_confusion.png)

## 7. El hallazgo: aquí el umbral no decide nada

El módulo 5 dice que el umbral es una decisión de negocio. En este modelo no lo es, y la
razón se ve en las probabilidades que predice.

In [8]:
print(f"probabilidades entre {probability.min():.3f} y {probability.max():.3f}")
print(f"casos entre 0,28 y 0,56: {int(probability.between(0.28, 0.56).sum())}")
print()
for cut in [0.30, 0.40, 0.50, 0.55]:
    marked = (probability >= cut).astype(int)
    hits = int(((marked == 1) & (test.verified == 1)).sum())
    print(f"umbral {cut:.2f}: marca {int(marked.sum())} cuentas, acierta {hits}")

probabilidades entre 0.204 y 0.651
casos entre 0,28 y 0,56: 0

umbral 0.30: marca 387 cuentas, acierta 248
umbral 0.40: marca 387 cuentas, acierta 248
umbral 0.50: marca 387 cuentas, acierta 248
umbral 0.55: marca 387 cuentas, acierta 248


![Las dos poblaciones de probabilidades](03_figures/06_probabilidades.png)

Las predicciones caen en dos montones separados por una banda vacía, así que mover el corte
dentro de ella no cambia ni una decisión. El modelo se ha convertido en la regla «¿es una
reclamación?» con dos pasos de aritmética por el medio.

## 8. Conclusión

Con un AUC de 0,697 y una de cada tres alarmas falsas, **esto no es un clasificador que se
pueda poner a decidir solo**. Lo que sí deja son dos cosas útiles: la confirmación de que
lo que separa a los dos grupos es el tipo de contenido y no la actividad de la cuenta, y la
tubería completa lista para el Curso 5, donde hay que predecir justamente esa variable.

El informe está en `04_reports/executive_summary.md` y los números publicados en
`04_reports/model_results.json`.